# Real dataset: Cdataset for drug repositioning

## Highlights:

- Cdataset is widely used in drug repositioning dataset and it is treated as one benchmark dataset. As compareable to existing methods, we used the same one from DRIMC, where the Cdataset includes 2353 known drug-disease associations involving **658 drugs** (as columns) and **409 diseases** (as rows).

- The data sources for drugs side information include drug chemical structure ($S_d^1$), Pfam domain annotation of drug targets ($S_d^2$) and gene ontology term of targets ($S_d^3$). 

- The data sources for disease include phenotype information ($S_t^1$) form OMIM database. 

## Remarks

- Cdataset is an expansion of Fdataset, where Fdataset is also a popular dataset with drug–disease associations compiled from DrugBank and OMIM. [Link](https://zenodo.org/records/8357512) is summarized by Berry et. al., showing different studies that used those datasets. 

## Workflow



---

## 1. Load in the dataset

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
PATH_TO_EXP = (
    "/Users/sijianfan/Documents/projects/BiSSGL/datasets/realAnalysis/cdataset"
)

In [3]:
df_Y = pd.read_table(os.path.join(PATH_TO_EXP, "c_admat_dgc.txt"), index_col=0)
print(f"The dataset has diseases and drugs: {df_Y.shape}")

The dataset has diseases and drugs: (409, 658)


---

## 2. Drug side information

### 1. Parse DrugBank XML

In [4]:
import xml.etree.ElementTree as ET
from collections import defaultdict


def parse_drugbank(xml_file, drugbank_ids):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    ns = {"db": "http://www.drugbank.ca"}

    drug_smiles = {}
    drug_targets = defaultdict(set)

    for drug in root.findall("db:drug", ns):
        db_id = drug.find("db:drugbank-id[@primary='true']", ns)
        if db_id is None:
            continue
        db_id = db_id.text

        if db_id not in drugbank_ids:
            continue

        # SMILES
        smiles = None
        for prop in drug.findall(".//db:property", ns):
            kind = prop.find("db:kind", ns)
            if kind is not None and kind.text == "SMILES":
                smiles = prop.find("db:value", ns).text
                break
        drug_smiles[db_id] = smiles

        # Targets (UniProt)
        for target in drug.findall("db:targets/db:target", ns):
            polypeptide = target.find("db:polypeptide", ns)
            if polypeptide is not None:
                uniprot = polypeptide.attrib.get("id")
                if uniprot:
                    drug_targets[db_id].add(uniprot)

    return drug_smiles, drug_targets

In [5]:
PATH_TO_XML = os.path.join(PATH_TO_EXP, "full database.xml")

In [6]:
drugbank_ids = df_Y.columns
drug_smiles, drug_targets = parse_drugbank(PATH_TO_XML, drugbank_ids)

### 2. Chemical structure → ECFP fingerprints

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem


def build_ecfp(drug_smiles, drug_index, n_bits=1024, radius=2):
    n_drugs = len(drug_index)
    X = np.zeros((n_drugs, n_bits), dtype=np.int8)

    for db_id, idx in drug_index.items():
        smiles = drug_smiles.get(db_id)
        if not smiles:
            continue
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
        X[idx, :] = np.array(fp)

    return X

Updated ECFP code (future-proof): Switch to MorganGenerator

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import numpy as np


def build_ecfp(drug_smiles, drug_index, n_bits=1024, radius=2):
    n_drugs = len(drug_index)
    X = np.zeros((n_drugs, n_bits), dtype=np.int8)

    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)

    for db_id, idx in drug_index.items():
        smiles = drug_smiles.get(db_id)
        if not smiles:
            continue

        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            continue

        fp = generator.GetFingerprint(mol)
        X[idx, :] = np.array(fp)

    return X

In [53]:
drugbank_ids = df_Y.columns
drug_index = {db_id: i for i, db_id in enumerate(drugbank_ids)}

U_ecfp = build_ecfp(drug_smiles, drug_index, n_bits=1024, radius=2)

In [ ]:
U_ecfp

array([[0, 1, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int8)

In [ ]:
print(U_ecfp.shape)  # should be (n_drugs, 1024)
print(U_ecfp.sum(axis=1)[:10])  # fingerprint density

(658, 1024)
[115  94  71   0 126  12  25  25  46  56]


### 3. Pfam domain features
#### 3.1 Load UniProt → Pfam mapping

In [35]:
def load_uniprot_pfam(pfam_file):
    uniprot2pfam = defaultdict(set)
    with open(pfam_file) as f:
        for line in f:
            if line.startswith("#"):
                continue
            fields = line.strip().split("\t")
            uniprot, pfam = fields[0], fields[4]
            uniprot2pfam[uniprot].add(pfam)
    return uniprot2pfam

In [38]:
pfam_file = os.path.join(PATH_TO_EXP, "Pfam-A.regions.tsv")
uniprot2pfam = load_uniprot_pfam(pfam_file)

### 3.2 Build drug–Pfam matrix

In [40]:
def build_pfam_matrix(drug_targets, uniprot2pfam, drug_index, min_freq=2):
    # collect domains
    pfam_counts = defaultdict(int)
    drug_pfam = defaultdict(set)

    for drug, prots in drug_targets.items():
        for p in prots:
            for pfam in uniprot2pfam.get(p, []):
                drug_pfam[drug].add(pfam)
                pfam_counts[pfam] += 1

    # filter domains
    pfams = [p for p, c in pfam_counts.items() if c >= min_freq]
    pfam_index = {p: i for i, p in enumerate(pfams)}

    X = np.zeros((len(drug_index), len(pfams)), dtype=np.int8)

    for drug, pfams in drug_pfam.items():
        if drug not in drug_index:
            continue
        i = drug_index[drug]
        for p in pfams:
            if p in pfam_index:
                X[i, pfam_index[p]] = 1

    return X, pfam_index

In [ ]:
U_pfam, pfam_index = build_pfam_matrix(
    drug_targets=drug_targets,
    uniprot2pfam=uniprot2pfam,
    drug_index=drug_index,
    min_freq=2,
)

In [44]:
print(U_pfam.shape)
print("Avg Pfam domains per drug:", U_pfam.sum(axis=1).mean())
print("Drugs with no Pfam domains:", (U_pfam.sum(axis=1) == 0).sum())
print("Most common Pfam domain count:", U_pfam.sum(axis=0).max())

(658, 416)
Avg Pfam domains per drug: 4.396656534954407
Drugs with no Pfam domains: 22
Most common Pfam domain count: 260


### 4. GO term features
#### 4.1 Load UniProt → GO

In [ ]:
def load_uniprot_go(go_file):
    uniprot2go = defaultdict(set)
    with open(go_file) as f:
        for line in f:
            if line.startswith("!"):
                continue
            fields = line.strip().split("\t")
            uniprot = fields[1]
            go_id = fields[4]
            uniprot2go[uniprot].add(go_id)
    return uniprot2go

Split the GO by 3 aspects

In [ ]:
from collections import defaultdict


def load_uniprot_go(go_file):
    """
    Returns:
        uniprot2go = {
            'BP': {uniprot: set(GO terms)},
            'MF': {uniprot: set(GO terms)},
            'CC': {uniprot: set(GO terms)},
        }
    """
    uniprot2go = {
        "BP": defaultdict(set),
        "MF": defaultdict(set),
        "CC": defaultdict(set),
    }

    aspect_map = {
        "P": "BP",
        "F": "MF",
        "C": "CC",
    }

    with open(go_file, "r") as f:
        for line in f:
            if line.startswith("!"):
                continue

            fields = line.rstrip("\n").split("\t")

            uniprot = fields[1]
            go_id = fields[4]
            aspect = fields[8]

            if aspect in aspect_map:
                uniprot2go[aspect_map[aspect]][uniprot].add(go_id)

    return uniprot2go

In [46]:
go_file = os.path.join(PATH_TO_EXP, "goa_human.gaf")
uniprot2go = load_uniprot_go(go_file)

In [ ]:
for aspect in ["BP", "MF", "CC"]:
    print(aspect, "proteins annotated:", len(uniprot2go[aspect]))

BP proteins annotated: 17791
MF proteins annotated: 18290
CC proteins annotated: 19019


In [48]:
from collections import defaultdict

drug_go = {
    "BP": defaultdict(set),
    "MF": defaultdict(set),
    "CC": defaultdict(set),
}

for drug, prots in drug_targets.items():
    for p in prots:
        for aspect in ["BP", "MF", "CC"]:
            if p in uniprot2go[aspect]:
                drug_go[aspect][drug].update(uniprot2go[aspect][p])

#### 4.2 Build drug–GO matrix

In [ ]:
def build_go_matrix(drug_targets, uniprot2go, drug_index, min_freq=3):
    go_counts = defaultdict(int)
    drug_go = defaultdict(set)

    for drug, prots in drug_targets.items():
        for p in prots:
            for go in uniprot2go.get(p, []):
                drug_go[drug].add(go)
                go_counts[go] += 1

    go_terms = [g for g, c in go_counts.items() if c >= min_freq]
    go_index = {g: i for i, g in enumerate(go_terms)}

    X = np.zeros((len(drug_index), len(go_terms)), dtype=np.int8)

    for drug, gos in drug_go.items():
        if drug not in drug_index:
            continue
        i = drug_index[drug]
        for g in gos:
            if g in go_index:
                X[i, go_index[g]] = 1

    return X, go_index

Split by 3 aspects

In [ ]:
import numpy as np
from collections import Counter


def build_go_matrix(drug_go_aspect, drug_index, min_freq=5, max_frac=0.8):
    """
    drug_go_aspect: dict {drug: set(GO terms)}
    """
    go_counts = Counter()
    for gos in drug_go_aspect.values():
        go_counts.update(gos)

    N = len(drug_index)

    go_terms = [go for go, c in go_counts.items() if min_freq <= c <= max_frac * N]

    go_index = {go: j for j, go in enumerate(sorted(go_terms))}

    X = np.zeros((N, len(go_index)), dtype=np.int8)

    for drug, gos in drug_go_aspect.items():
        if drug not in drug_index:
            continue
        i = drug_index[drug]
        for go in gos:
            if go in go_index:
                X[i, go_index[go]] = 1

    return X, go_index

In [ ]:
U_go_bp, go_bp_index = build_go_matrix(drug_go["BP"], drug_index, min_freq=5)

U_go_mf, go_mf_index = build_go_matrix(drug_go["MF"], drug_index, min_freq=5)

U_go_cc, go_cc_index = build_go_matrix(drug_go["CC"], drug_index, min_freq=5)

In [ ]:
print("GO:BP", U_go_bp.shape, "avg terms:", U_go_bp.sum(axis=1).mean())

print("GO:MF", U_go_mf.shape, "avg terms:", U_go_mf.sum(axis=1).mean())

print("GO:CC", U_go_cc.shape, "avg terms:", U_go_cc.sum(axis=1).mean())

GO:BP (658, 1718) avg terms: 64.24468085106383
GO:MF (658, 566) avg terms: 23.477203647416413
GO:CC (658, 296) avg terms: 19.817629179331306


In [54]:
U_blocks = {
    "ecfp": U_ecfp,
    "pfam": U_pfam,
    "go_bp": U_go_bp,
    "go_mf": U_go_mf,
    "go_cc": U_go_cc,
}

### 5. Save out the U side information

In [ ]:
U = np.hstack(
    [
        U_blocks["ecfp"],
        U_blocks["pfam"],
        U_blocks["go_bp"],
        U_blocks["go_mf"],
        U_blocks["go_cc"],
    ]
)

In [60]:
print(U.shape)

(658, 4020)


In [61]:
feature_names = []

# ECFP
feature_names += [f"ECFP_{i}" for i in range(U_blocks["ecfp"].shape[1])]

# Pfam
feature_names += [f"PFAM_{pfam}" for pfam in pfam_index]

# GO
feature_names += [f"GO_BP_{go}" for go in go_bp_index]
feature_names += [f"GO_MF_{go}" for go in go_mf_index]
feature_names += [f"GO_CC_{go}" for go in go_cc_index]

In [ ]:
# Ensure correct drug order
drug_ids_ordered = [d for d, _ in sorted(drug_index.items(), key=lambda x: x[1])]

df_U = pd.DataFrame(U, index=drug_ids_ordered, columns=feature_names)

In [64]:
df_Y.columns

Index(['DB00014', 'DB00035', 'DB00091', 'DB00104', 'DB00115', 'DB00122',
       'DB00125', 'DB00126', 'DB00131', 'DB00136',
       ...
       'DB08801', 'DB08802', 'DB08804', 'DB08820', 'DB08824', 'DB08835',
       'DB08896', 'DB08901', 'DB08906', 'DB08907'],
      dtype='object', length=658)

In [67]:
df_U.index

Index(['DB00014', 'DB00035', 'DB00091', 'DB00104', 'DB00115', 'DB00122',
       'DB00125', 'DB00126', 'DB00131', 'DB00136',
       ...
       'DB08801', 'DB08802', 'DB08804', 'DB08820', 'DB08824', 'DB08835',
       'DB08896', 'DB08901', 'DB08906', 'DB08907'],
      dtype='object', length=658)

In [ ]:
output_path = os.path.join(PATH_TO_EXP, "drugs_features.csv")
df_U.to_csv(output_path)

---

## 3. Diseases side information as features

In [4]:
df_Y.index

Index(['D102100', 'D102300', 'D102400', 'D102500', 'D103100', 'D103230',
       'D103285', 'D103780', 'D104130', 'D104300',
       ...
       'D608232', 'D608266', 'D608320', 'D608437', 'D608456', 'D608583',
       'D608622', 'D608636', 'D608895', 'D608907'],
      dtype='object', length=409)

1.	HPO (Human Phenotype Ontology) binary matrix — best surrogate for MimMiner.
OMIM ↔ HPO mappings exist (Monarch/OMIM annotations). Binary presence of HPO terms per disease; propagate to ancestors to reduce sparsity.
2.	OMIM text TF–IDF → low-dim embedding — captures free-text phenotype descriptions (MimMiner is text-based). Use TF–IDF → truncated SVD (LSA) or PCA → treat components as features.
3.	Disease–gene matrix (you already planned via OMIM) — mechanistic and essential.
4.	Pathway / Reactome / KEGG aggregation of disease genes — lower-dim and often highly predictive.
5.	Similarity-derived embeddings — convert pairwise MimMiner-like similarity into features via MDS / spectral embedding / diffusion maps. Useful if you already compute a pairwise similarity.
6.	Disease ontology / MeSH / DO terms — hierarchical disease categories as features (coarser than HPO).

Combine a subset (1,2,3,4) for best performance & interpretability.

### 1. Parse OMIM JSON

In [ ]:
import os

OMIM_CACHE_DIR = os.path.join(PATH_TO_EXP, "./omim_cache")
os.makedirs(OMIM_CACHE_DIR, exist_ok=True)

In [ ]:
import requests
import json
import time


def query_omim_entry(omim_id, api_key, cache_dir=OMIM_CACHE_DIR, sleep_time=0.5):
    cache_file = os.path.join(cache_dir, f"{omim_id}.json")

    # use cached version if exists
    if os.path.exists(cache_file):
        with open(cache_file, "r") as f:
            return json.load(f)

    url = "https://api.omim.org/api/entry"
    params = {
        "mimNumber": omim_id,
        "format": "json",
        "apiKey": api_key,
        "include": "geneMap,clinicalSynopsis",
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print(f"OMIM API error for {omim_id}: {response.status_code}")
        return None

    data = response.json()

    with open(cache_file, "w") as f:
        json.dump(data, f, indent=2)

    time.sleep(sleep_time)  # be nice to OMIM servers
    return data

In [ ]:
disease_to_omim = {d: [d[1:]] for d in df_Y.index}  # remove leading 'D'

In [21]:
omim_api_key = "STw1icyaR3mDxZYTJWQc5w"

unique_omim_ids = sorted({v[0] for v in disease_to_omim.values()})

omim_raw = {}
for omim_id in unique_omim_ids:
    omim_raw[omim_id] = query_omim_entry(omim_id, omim_api_key)

#### 1.1 Parse OMIM JSON into disease-level dictionaries

In [ ]:
from collections import defaultdict


def parse_omim_raw(disease_to_omim, omim_raw):
    """
    Returns:
        disease_genes: dict {disease_id: set(gene symbols)}
        disease_text: dict {disease_id: string}
    """
    disease_genes = defaultdict(set)
    disease_text = {}

    for disease_id, omim_ids in disease_to_omim.items():
        texts = []

        for omim_id in omim_ids:
            raw = omim_raw.get(omim_id)
            if raw is None:
                continue

            try:
                entry = raw["omim"]["entryList"][0]["entry"]
            except (KeyError, IndexError):
                continue

            # ---- genes ----
            gene_map = entry.get("geneMap")
            if gene_map:
                gene_symbols = gene_map.get("geneSymbols")
                if gene_symbols:
                    # geneSymbols may be "BRCA1, BRCA1-AS1"
                    for g in gene_symbols.replace(";", ",").split(","):
                        disease_genes[disease_id].add(g.strip())

            # ---- text ----
            clin = entry.get("clinicalSynopsis")
            if clin:
                texts.append(" ".join(str(v) for v in clin.values()))

        # merge text from multiple OMIM IDs
        if texts:
            disease_text[disease_id] = " ".join(texts)
        else:
            disease_text[disease_id] = ""

    return disease_genes, disease_text

In [ ]:
disease_genes, disease_text = parse_omim_raw(
    disease_to_omim=disease_to_omim, omim_raw=omim_raw
)

In [54]:
# how many diseases have at least one gene?
sum(len(v) > 0 for v in disease_genes.values()), "/", len(disease_genes)

(73, '/', 73)

In [27]:
# average genes per disease
import numpy as np

np.mean([len(v) for v in disease_genes.values()])

1.3835616438356164

In [28]:
# example
list(disease_genes.items())[:3]

[('D102300', {'RLS1'}), ('D107320', {'ATPLS'}), ('D109200', {'AGA1', 'MPB'})]

### 2. Map gene symbols → Entrez IDs

In [29]:
!pip install mygene

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [mygene]2m3/6 [httpx]


In [ ]:
import mygene

mg = mygene.MyGeneInfo()


def map_symbols_to_entrez(disease_genes):
    disease_entrez = defaultdict(set)

    all_symbols = sorted({g for genes in disease_genes.values() for g in genes})
    res = mg.querymany(
        all_symbols, scopes="symbol", fields="entrezgene", species="human"
    )

    sym2entrez = {r["query"]: str(r["entrezgene"]) for r in res if "entrezgene" in r}

    for d, genes in disease_genes.items():
        for g in genes:
            if g in sym2entrez:
                disease_entrez[d].add(sym2entrez[g])

    return disease_entrez

In [59]:
disease_entrez = map_symbols_to_entrez(disease_genes)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
1 input query terms found dup hits:	[('GULOP', 2)]
35 input query terms found no hit:	['AGA1', 'AORF', 'ATOD4', 'AUTS4', 'C15DUPq11-q13', 'C17DELp13.3', 'CMCT', 'CXDELq21', 'D13S25', 'DB


In [72]:
# coverage
print(
    sum(len(v) > 0 for v in disease_entrez.values()),
    "/",
    len(disease_entrez),
)

# average number of genes per disease
import numpy as np

print(
    "Avg Entrez genes per disease:", np.mean([len(v) for v in disease_entrez.values()])
)

# example entries
list(disease_entrez.items())[:3]

65 / 65
Avg Entrez genes per disease: 1.0153846153846153


[('D102300', {'192142'}), ('D107320', {'100499532'}), ('D109350', {'59330'})]

### 3. Build the disease–gene feature matrix V_gene

In [73]:
from collections import Counter

gene_counts = Counter()
for genes in disease_entrez.values():
    gene_counts.update(genes)

N_diseases = len(disease_entrez)

# frequency filtering (recommended)
genes_kept = [g for g, c in gene_counts.items() if c >= 1 and c <= 0.8 * N_diseases]

print("Total genes:", len(gene_counts))
print("Genes kept after filtering:", len(genes_kept))

Total genes: 66
Genes kept after filtering: 66


In [ ]:
gene_index = {g: j for j, g in enumerate(sorted(genes_kept))}

In [75]:
disease_ids = df_Y.index
disease_index = {omim_id: i for i, omim_id in enumerate(disease_ids)}

In [ ]:
import numpy as np

V_gene = np.zeros((len(disease_index), len(gene_index)), dtype=np.int8)

for disease, genes in disease_entrez.items():
    if disease not in disease_index:
        continue

    i = disease_index[disease]
    for g in genes:
        if g in gene_index:
            V_gene[i, gene_index[g]] = 1

In [77]:
print(V_gene.shape)
print("Avg genes per disease:", V_gene.sum(axis=1).mean())
print("Diseases with no genes:", (V_gene.sum(axis=1) == 0).sum())

(409, 66)
Avg genes per disease: 0.16136919315403422
Diseases with no genes: 344


### 4. Phenotype features

In [ ]:
from collections import defaultdict


def load_omim_hpo(hpoa_file):
    """
    Returns:
        omim2hpo: dict { '154700' : set(HP terms) }
    """
    omim2hpo = defaultdict(set)

    with open(hpoa_file, "r") as f:
        for line in f:
            if line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t")

            disease_id = fields[0]  # e.g. OMIM:154700
            hpo_id = fields[3]  # e.g. HP:0001250

            if disease_id.startswith("OMIM:"):
                omim = disease_id.split(":")[1]
                omim2hpo[omim].add(hpo_id)

    return omim2hpo

In [80]:
hpoa_file = os.path.join(PATH_TO_EXP, "phenotype.hpoa")

omim2hpo = load_omim_hpo(hpoa_file)

In [89]:
disease_hpo = defaultdict(set)

for disease, omim_ids in disease_to_omim.items():
    for omim in omim_ids:
        if omim in omim2hpo:
            disease_hpo[disease].update(omim2hpo[omim])

In [90]:
print(sum(len(v) > 0 for v in disease_hpo.values()), "/", len(disease_index))

352 / 409


In [ ]:
from collections import defaultdict


def load_hpo_ancestors(obo_file):
    """
    Returns:
        ancestors: dict { HPO_term -> set(all ancestor HPO terms) }
    """
    parents = defaultdict(set)

    # ---- Step 1: parse parents ----
    current = None
    with open(obo_file, "r") as f:
        for line in f:
            line = line.strip()
            if line == "[Term]":
                current = None
            elif line.startswith("id:"):
                current = line.split("id:")[1].strip()
            elif line.startswith("is_a:") and current:
                parent = line.split("is_a:")[1].split("!")[0].strip()
                parents[current].add(parent)

    # ---- Step 2: compute ancestors with memoization ----
    ancestors = {}

    def get_ancestors(h):
        if h in ancestors:
            return ancestors[h]

        anc = set()
        for p in parents.get(h, []):
            anc.add(p)
            anc.update(get_ancestors(p))

        ancestors[h] = anc
        return anc

    # IMPORTANT: freeze the list of HPO terms
    all_terms = list(parents.keys())

    for h in all_terms:
        get_ancestors(h)

    return ancestors

In [95]:
obo_file = os.path.join(PATH_TO_EXP, "hp.obo")

hpo_ancestors = load_hpo_ancestors(obo_file)

In [96]:
len(hpo_ancestors)
list(hpo_ancestors.items())[:3]

[('HP:0000001', set()),
 ('HP:0000118', {'HP:0000001'}),
 ('HP:0001507', {'HP:0000001', 'HP:0000118'})]

In [97]:
# propagate HPO terms
disease_hpo_expanded = {}

for d, terms in disease_hpo.items():
    expanded = set(terms)
    for t in terms:
        expanded.update(hpo_ancestors.get(t, set()))
    disease_hpo_expanded[d] = expanded

In [ ]:
from collections import Counter
import numpy as np

hpo_counts = Counter()
for terms in disease_hpo_expanded.values():
    hpo_counts.update(terms)

N = len(disease_index)

hpo_terms = [h for h, c in hpo_counts.items() if c >= 2 and c <= 0.9 * N]

hpo_index = {h: j for j, h in enumerate(sorted(hpo_terms))}

In [ ]:
V_hpo = np.zeros((len(disease_index), len(hpo_index)), dtype=np.int8)

for disease, terms in disease_hpo_expanded.items():
    i = disease_index[disease]
    for h in terms:
        if h in hpo_index:
            V_hpo[i, hpo_index[h]] = 1

In [109]:
print("V_hpo shape:", V_hpo.shape)
print("Avg HPO terms per disease:", V_hpo.sum(axis=1).mean())
print("Diseases with no HPO terms:", (V_hpo.sum(axis=1) == 0).sum())

V_hpo shape: (409, 1794)
Avg HPO terms per disease: 42.28850855745721
Diseases with no HPO terms: 57


In [110]:
V_hpo

array([[1, 1, 1, ..., 0, 0, 0],
       [1, 0, 1, ..., 0, 0, 0],
       [1, 0, 1, ..., 0, 0, 0],
       ...,
       [1, 0, 1, ..., 1, 0, 0],
       [1, 0, 1, ..., 0, 0, 0],
       [1, 0, 1, ..., 0, 0, 0]], dtype=int8)

In [111]:
V_blocks = {
    "gene": V_gene,
    "hpo": V_hpo,
}

---

### 5. Save out the V side information

In [ ]:
import numpy as np

V = np.hstack([V_gene, V_hpo])  # (409 × n_gene)  # (409 × n_hpo)

In [115]:
feature_names_V = []

# gene features
feature_names_V += [f"GENE_{g}" for g in gene_index]

# HPO features
feature_names_V += [f"HPO_{h}" for h in hpo_index]

assert len(feature_names_V) == V.shape[1]

In [ ]:
import pandas as pd

disease_ids_ordered = [d for d, _ in sorted(disease_index.items(), key=lambda x: x[1])]

df_V = pd.DataFrame(V, index=disease_ids_ordered, columns=feature_names_V)

In [118]:
df_V.shape

(409, 1860)

In [ ]:
output_path = os.path.join(PATH_TO_EXP, "diseases_features.csv")
df_V.to_csv(output_path)